# cli

> CLI tool for declarative capability management

In [ ]:
#| default_exp cli

In [ ]:
#| export
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, Any, Dict, Optional, List

import typer
import yaml

from cjm_substrate import __version__ as _substrate_version
from cjm_substrate.core.config import (
    load_config, set_config, get_config, CondaType, RuntimeMode
)
from cjm_substrate.core.platform import (
    run_shell_command, conda_env_exists, get_conda_command, build_conda_command,
    ensure_runtime_available, download_micromamba, get_micromamba_binary_path,
    get_current_platform
)
from cjm_substrate.core.metadata import ResourceRequirements
from cjm_substrate.core.manifest_format import (
    ManifestV2, InstallSection, CodeSection, DriftTracking,
    CURRENT_FORMAT_VERSION,
    load_manifest, write_manifest, manifest_to_dict, compute_config_schema_hash,
    compute_structural_surface_hash,
)

app = typer.Typer(help="cjm-substrate CLI", no_args_is_help=True)

@app.callback()
def main(
    ctx:typer.Context,
    cjm_config:Annotated[Optional[Path], typer.Option(
        "--cjm-config",
        help="Path to cjm.yaml configuration file"
    )]=None,
    data_dir:Annotated[Optional[Path], typer.Option(
        "--data-dir",
        help="Override data directory (manifests, logs)"
    )]=None,
    conda_prefix:Annotated[Optional[Path], typer.Option(
        "--conda-prefix",
        help="Override conda/mamba prefix path"
    )]=None,
    conda_type:Annotated[Optional[str], typer.Option(
        "--conda-type",
        help="Conda implementation: micromamba, miniforge, or conda"
    )]=None,
) -> None:
    """cjm-substrate CLI for managing isolated capability environments."""
    cfg = load_config(
        config_path=cjm_config,
        data_dir=data_dir,
        conda_prefix=conda_prefix,
        conda_type=conda_type
    )
    set_config(cfg)

## Setup Runtime

The `setup-runtime` command downloads and installs micromamba for project-local mode. This is required before running `install-all` when using `conda_type: micromamba` in the configuration.

In [ ]:
#| export
@app.command("setup-runtime")
def setup_runtime(
    force:bool=typer.Option(False, "--force", "-f", help="Re-download even if binary exists")
) -> None:
    """Download and setup micromamba runtime for project-local mode."""
    cfg = get_config()
    
    # Check if micromamba mode
    if cfg.runtime.conda_type != CondaType.MICROMAMBA:
        typer.echo(f"Runtime setup is only needed for micromamba mode.")
        typer.echo(f"Current conda_type: {cfg.runtime.conda_type.value}")
        typer.echo(f"Set 'conda_type: micromamba' in cjm.yaml to use project-local runtime.")
        raise typer.Exit(code=0)
    
    # Determine binary path
    binary_path = get_micromamba_binary_path(cfg)
    
    if binary_path is None:
        typer.echo("Error: Cannot determine micromamba binary path.", err=True)
        typer.echo("Set 'runtime.prefix' or 'runtime.binaries' in cjm.yaml.", err=True)
        raise typer.Exit(code=1)
    
    # Check if already exists
    if binary_path.exists() and not force:
        typer.echo(f"Micromamba already installed at: {binary_path}")
        typer.echo(f"Use --force to re-download.")
        raise typer.Exit(code=0)
    
    # Show what will be done
    platform_str = get_current_platform()
    typer.echo(f"=== Setup Runtime ===\n")
    typer.echo(f"Platform: {platform_str}")
    typer.echo(f"Destination: {binary_path}")
    typer.echo("")
    
    # Create runtime directory structure
    if cfg.runtime.prefix:
        (cfg.runtime.prefix / "bin").mkdir(parents=True, exist_ok=True)
        (cfg.runtime.prefix / "envs").mkdir(parents=True, exist_ok=True)
        typer.echo(f"Created runtime directories at: {cfg.runtime.prefix}")
    
    # Download micromamba
    success = download_micromamba(binary_path, platform_str, show_progress=True)
    
    if success:
        typer.echo(f"\nRuntime setup complete!")
        typer.echo(f"You can now run: cjm-ctl install-all --capabilities <capabilities.yaml>")
    else:
        typer.echo(f"\nError: Failed to download micromamba.", err=True)
        raise typer.Exit(code=1)

In [ ]:
#| export
def _check_runtime_available() -> None:
    """Check if the configured conda runtime is available, exit with helpful message if not."""
    cfg = get_config()
    
    if not ensure_runtime_available(cfg):
        if cfg.runtime.conda_type == CondaType.MICROMAMBA:
            typer.echo("Error: Micromamba runtime not found.", err=True)
            typer.echo("Run 'cjm-ctl setup-runtime' first to download micromamba.", err=True)
        elif cfg.runtime.conda_type == CondaType.MINIFORGE:
            typer.echo("Error: Mamba not found in PATH.", err=True)
            typer.echo("Install miniforge/mambaforge or set conda_type to 'conda'.", err=True)
        else:
            typer.echo("Error: Conda not found in PATH.", err=True)
            typer.echo("Install conda or set the correct conda_type in cjm.yaml.", err=True)
        raise typer.Exit(code=1)

In [ ]:
#| export
def _get_conda_cmd_str() -> str:
    """Get the conda/micromamba command string for shell commands."""
    cfg = get_config()
    cmd_parts = get_conda_command(cfg)
    return " ".join(cmd_parts)

In [ ]:
#| export
import tempfile
import urllib.request
import urllib.error

def _download_url_to_temp(
    url: str,  # URL to download
    suffix: str = ".yml"  # File suffix for temp file
) -> Optional[Path]:  # Path to temp file or None if failed
    """Download a URL to a temporary file. Returns None if download fails."""
    try:
        # Create a named temp file that won't be auto-deleted
        fd, temp_path = tempfile.mkstemp(suffix=suffix)
        os.close(fd)
        
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, temp_path)
        return Path(temp_path)
    except (urllib.error.URLError, urllib.error.HTTPError, IOError) as e:
        print(f"Failed to download {url}: {e}")
        return None


def _resolve_env_file(
    env_file: str  # Path or URL to environment file
) -> tuple[str, Optional[Path]]:  # (resolved_path, temp_file_to_cleanup)
    """Resolve env_file to a local path, downloading if it's a URL.
    
    Returns (local_path, temp_file) where temp_file is set if we created
    a temporary file that should be cleaned up later.
    """
    if env_file.startswith("http://") or env_file.startswith("https://"):
        temp_file = _download_url_to_temp(env_file)
        if temp_file:
            return str(temp_file), temp_file
        else:
            raise RuntimeError(f"Failed to download environment file: {env_file}")
    else:
        # Local path, no cleanup needed
        return env_file, None

In [ ]:
#| export
def run_cmd(
    cmd: str,  # Shell command to execute
    check: bool = True  # Whether to raise on non-zero exit
) -> None:
    """Run a shell command and stream output.
    
    Uses the platform's default shell (no hardcoded /bin/bash).
    """
    run_shell_command(cmd, check=check)

In [ ]:
#| export
def _generate_manifest(
    env_name: str,  # Name of the Conda environment
    package_name: str,  # Package source string (git URL or package name)
    manifest_dir: Path  # Directory to write manifest JSON files
) -> Optional[Path]:  # Path to the manifest written, or None on failure
    """Run introspection script inside the target env and write a v2.0 manifest.
    
    Builds a `ManifestV2` from the introspection output + install-time markers,
    computes `drift_tracking.config_schema_hash`, and writes via
    `write_manifest` (nested layout, indent=2). Both `installed_at` and
    `regenerated_at` are set to "now"; `regenerate_manifest` preserves the
    original `installed_at` via post-write fix-up.
    """
    print(f"[{env_name}] Generating manifest...")
    cfg = get_config()
    
    # Robust Module Name Extraction
    if package_name.startswith("git+"):
        # Case: git+https://github.com/cj-mills/repo-name.git
        url = package_name[4:]
        repo_part = url.split('/')[-1]
        module_name = repo_part.split('.git')[0].replace('-', '_')
    elif package_name.startswith("-e "):
        path = package_name.split("-e ")[1].strip()
        module_name = Path(path).name.replace('-', '_')
    elif "/" in package_name or "\\" in package_name:
        module_name = Path(package_name).name.replace('-', '_')
    else:
        module_name = package_name.replace('-', '_')

    print(f"[{env_name}] Introspecting module: {module_name}")
    
    # Enhanced introspection script. The script:
    # 1. Derives identity from the installed distribution + capability class.
    # 2. Instantiates the capability class to get config_schema
    # 3. Merges config_schema into metadata (if not already present)
    introspection_script = f'''
import json
import importlib
# NEW-STYLE (Option C / PILLAR 1c): every capability is re-based — identity is
# derived entirely from the installed distribution + the discovered capability
# class (no meta module). type / resources / db_path are intentionally OMITTED —
# task comes from bound adapters, admission measures resources empirically, and
# db_path is an adapter-derived persistence concern.
import sys as _sys
import importlib.metadata as _md
import inspect as _inspect
from cjm_substrate.core.capability import ToolCapability
meta = {{}}
_pmod = importlib.import_module("{module_name}.capability")
_cands = [obj for _n, obj in _inspect.getmembers(_pmod, _inspect.isclass)
          if issubclass(obj, ToolCapability) and obj is not ToolCapability
          and not _inspect.isabstract(obj)
          and obj.__module__ == _pmod.__name__]
if len(_cands) != 1:
    raise SystemExit(
        "[introspection] new-style capability discovery expected exactly 1 "
        "concrete ToolCapability subclass in " + _pmod.__name__ + ", found "
        + str(len(_cands)) + ": " + str([c.__name__ for c in _cands])
        + ". Declare a pyproject [project.entry-points] entry to disambiguate.")
_cap = _cands[0]
_dist_list = _md.packages_distributions().get("{module_name}")
_dist = _dist_list[0] if _dist_list else "{module_name}".replace("_", "-")
_dm = _md.metadata(_dist)
meta["name"] = _dm.get("Name", _dist)
meta["version"] = _dm.get("Version", "") or ""
meta["description"] = _dm.get("Summary", "") or ""
meta["module"] = _pmod.__name__
meta["class"] = _cap.__name__
meta["python_path"] = _sys.executable  # the worker-env interpreter (proxy spawns it)

# Try to get config_schema from capability instance if not in metadata
if "config_schema" not in meta:
    try:
        # Import capability module and class
        capability_module = meta.get("module", "{module_name}.capability")
        capability_class = meta.get("class", "")
        
        if capability_module and capability_class:
            mod = importlib.import_module(capability_module)
            cls = getattr(mod, capability_class)
            
            # Instantiate and get config schema
            instance = cls()
            if hasattr(instance, "get_config_schema"):
                meta["config_schema"] = instance.get_config_schema()
            
            # Clean up
            if hasattr(instance, "cleanup"):
                try:
                    instance.cleanup()
                except:
                    pass
    except Exception as e:
        # Config schema extraction is optional - continue without it
        pass

# CR-12: introspect the capability's WORKER_ENV spawn-env contract. Read off the
# CLASS (no instantiation) so capabilities that can't construct without resources
# still surface their env contract. EnvVarSpec is a flat dataclass; asdict gives
# JSON-serializable entries the substrate stores in the manifest code section.
try:
    import dataclasses as _dc
    _pm = meta.get("module", "{module_name}.capability")
    _pc = meta.get("class", "")
    if _pm and _pc:
        _wmod = importlib.import_module(_pm)
        _wcls = getattr(_wmod, _pc)
        _worker_env = getattr(_wcls, "WORKER_ENV", None) or []
        if _worker_env:
            meta["worker_env"] = [
                _dc.asdict(s) if _dc.is_dataclass(s) else dict(s) for s in _worker_env
            ]
except Exception:
    pass

# Pass-2 Thread 3: record the capability's STRUCTURAL SURFACE (public
# methods + signatures + properties + class attributes) by pure
# self-introspection — zero protocol/adapter knowledge. Read off the
# CLASS (no instantiation). Tolerant import: a capability env still on a
# pre-fracture substrate has no core.capability — the manifest simply
# omits the surface until the env resyncs (install-all --force).
try:
    from cjm_substrate.core.capability import derive_structural_surface
    _sm = meta.get("module", "{module_name}.capability")
    _sc = meta.get("class", "")
    if _sm and _sc:
        _smod = importlib.import_module(_sm)
        _scls = getattr(_smod, _sc)
        meta["structural_surface"] = derive_structural_surface(_scls)
except Exception:
    pass

print(json.dumps(meta, indent=2))
'''
    
    # Build environment with CJM paths for capability introspection
    env = dict(os.environ)
    env["CJM_CAPABILITY_DATA_DIR"] = str(cfg.capability_data_dir)
    if cfg.models_dir:
        env["CJM_MODELS_DIR"] = str(cfg.models_dir)
    
    # SG-2: write the introspection script to a temp .py file rather than
    # passing it inline via `python -c '...'`. cmd.exe does not honor POSIX
    # single quotes, so the inline form silently breaks Windows installs.
    # delete=False because we must close the file before another process
    # (the conda-env python interpreter) reads it on Windows; the finally
    # block unlinks it.
    introspection_fd, introspection_path = tempfile.mkstemp(
        prefix=f"cjm_introspect_{env_name}_",
        suffix=".py",
    )
    try:
        with os.fdopen(introspection_fd, "w") as f:
            f.write(introspection_script)
        
        # The introspection command - use configurable conda command.
        # Double-quote the script path so paths with spaces (common on
        # Windows) work in either shell.
        conda_cmd = _get_conda_cmd_str()
        introspection_cmd = f'{conda_cmd} run -n {env_name} python "{introspection_path}"'
        
        try:
            # Check output, capture stdout
            # Use shell=True without explicit executable for cross-platform support
            # Pass env to propagate CJM paths to the introspection script
            result = subprocess.run(
                introspection_cmd,
                shell=True,
                capture_output=True,
                text=True,
                check=True,
                env=env
            )
            result_str = result.stdout.strip()
            
            # Robust JSON Parsing: 
            # Sometimes 'conda run' leaks warnings into stdout. We try to find the JSON block.
            try:
                start = result_str.find('{')
                end = result_str.rfind('}') + 1
                if start != -1 and end != 0:
                    json_str = result_str[start:end]
                    meta_json = json.loads(json_str)
                else:
                    # Fallback to full string if brackets not found
                    meta_json = json.loads(result_str)
            except json.JSONDecodeError as e:
                print(f"ERROR: Failed to parse JSON from introspection output.")
                print(f"Raw Output:\n{result_str}")
                return None

            # SG-34: drop the dead `type` field from new manifests.
            meta_json.pop("type", None)

            capability_name = meta_json.get('name', 'unknown')
            out_file = manifest_dir / f"{capability_name}.json"
            now_iso = datetime.now(timezone.utc).isoformat()

            # CR-8: convert flat introspection output into a nested ManifestV2.
            # The introspection script emits a flat dict with both code-section
            # fields (name/version/description/module/class/resources/
            # config_schema) AND install-section fields the capability owns
            # (python_path/db_path/env_vars). Substrate adds the installer-side
            # install fields (installed_at/installer_version/package_source/
            # conda_env) here.

            # Parse the resources sub-dict into a typed object (or None when the
            # capability predates Phase 5a). Substrate stores strings only — no
            # host-side imports of interface libraries needed.
            intro_res = meta_json.get("resources")
            res_obj: Optional[ResourceRequirements] = None
            if isinstance(intro_res, dict):
                res_obj = ResourceRequirements(
                    requires_gpu=bool(intro_res.get("requires_gpu", False)),
                    platforms=list(intro_res.get("platforms", []) or []),
                    accelerators=list(intro_res.get("accelerators", []) or []),
                )

            config_schema = meta_json.get("config_schema")
            # CR-12: worker-env contract (list of asdict(EnvVarSpec)); None for capabilities without one.
            intro_worker_env = meta_json.get("worker_env")
            # Pass-2 Thread 3: surface recorded in-env; None when the capability
            # env still runs a pre-fracture substrate (hash stays None so the
            # drift check skips rather than flagging every old install).
            intro_surface = meta_json.get("structural_surface")

            manifest = ManifestV2(
                install=InstallSection(
                    # Capability-author-supplied install-section fields
                    python_path=str(meta_json.get("python_path", "") or ""),
                    db_path=str(meta_json.get("db_path", "") or ""),
                    env_vars=dict(meta_json.get("env_vars", {}) or {}),
                    # Substrate-supplied install-section fields. conda_env now
                    # written here directly (no more install_all post-write).
                    conda_env=env_name,
                    installed_at=now_iso,
                    installer_version=f"cjm-ctl {_substrate_version}",
                    package_source=package_name,
                ),
                code=CodeSection(
                    name=str(meta_json.get("name", "") or ""),
                    version=str(meta_json.get("version", "") or ""),
                    description=str(meta_json.get("description", "") or ""),
                    module=str(meta_json.get("module", "") or ""),
                    class_name=str(meta_json.get("class", "") or ""),
                    resources=res_obj,
                    config_schema=config_schema,
                    regenerated_at=now_iso,
                    worker_env=intro_worker_env,
                    structural_surface=intro_surface,
                ),
                drift_tracking=DriftTracking(
                    config_schema_hash=compute_config_schema_hash(config_schema),
                    structural_surface_hash=(compute_structural_surface_hash(intro_surface)
                                             if intro_surface is not None else None),
                ),
                overrides={},
            )
            
            # Log detected metadata
            if config_schema is not None:
                print(f"[{env_name}] Config schema: captured")
            if intro_surface is not None:
                print(f"[{env_name}] Structural surface: "
                      f"{len(intro_surface.get('methods', []))} methods, "
                      f"{len(intro_surface.get('properties', []))} properties")
            
            write_manifest(out_file, manifest)
            print(f"[{env_name}] Wrote manifest to {out_file}")
            # SG-3: return the path so callers can update the manifest by exact
            # file rather than scanning `manifest_dir` with a substring match.
            return out_file
            
        except subprocess.CalledProcessError as e:
            print(f"Error generating manifest for {env_name}:")
            print(e.stderr if e.stderr else str(e))
            return None
        except Exception as e:
            print(f"Unexpected error generating manifest: {e}")
            return None
    finally:
        # Best-effort cleanup of the temp introspection script.
        try:
            os.unlink(introspection_path)
        except OSError:
            pass

## Regenerate Manifest

The `regenerate-manifest` command re-runs the introspection pipeline for an
already-installed capability and rewrites its manifest in place. Picks up new
substrate-side manifest fields (resources, install-time metadata) without
forcing a full `install-all --force`.

Package-source recovery order:
1. `--package <spec>` operator override (always wins when provided)
2. The manifest's own `package_source` field (post-Phase-5a manifests)
3. `--capabilities capabilities.yaml` lookup by capability name (legacy manifests without `package_source`)
4. Error with actionable message if none of the above resolve

The ecosystem-wide `cascade_manifests.py` (loops this command over every
capability) is deferred to CR-8's nested-format migration.

In [ ]:
#| export
@app.command("regenerate-manifest")
def regenerate_manifest(
    capability_name: str = typer.Argument(..., help="Capability name as it appears in the manifest"),
    capabilities_path: Optional[str] = typer.Option(
        None, "--capabilities",
        help="Path to capabilities.yaml for package_source recovery (legacy manifests)",
    ),
    package: Optional[str] = typer.Option(
        None, "--package",
        help="Package spec override (e.g., git URL or pip name); wins over manifest/capabilities.yaml lookups",
    ),
) -> None:
    """Re-run introspection for an installed capability and rewrite its manifest.
    
    Reads the existing manifest via `load_manifest`, recovers `env_name` +
    `package_source` from the install section, runs `_generate_manifest` to
    refresh the code section, then post-writes to preserve the original
    `installed_at` so the regenerate only updates `regenerated_at` semantically.
    Always emits v2.0 layout.
    """
    cfg = get_config()
    manifest_path = cfg.manifests_dir / f"{capability_name}.json"
    if not manifest_path.exists():
        typer.echo(f"No manifest found at {manifest_path}", err=True)
        raise typer.Exit(code=1)
    
    try:
        existing = load_manifest(manifest_path)
    except (json.JSONDecodeError, ValueError) as e:
        typer.echo(f"Cannot parse existing manifest {manifest_path}: {e}", err=True)
        raise typer.Exit(code=1)
    
    # Recover env_name: explicit field preferred, python_path fallback for legacy.
    env_name = existing.install.conda_env or _extract_env_from_python_path(
        existing.install.python_path
    )
    if not env_name:
        typer.echo(
            f"Cannot determine conda env for {capability_name!r}: manifest lacks both "
            f"'conda_env' field and a parseable 'python_path'. Reinstall the capability "
            f"via `cjm-ctl install-all` to refresh.",
            err=True,
        )
        raise typer.Exit(code=1)
    
    # Recover package_source: --package > manifest.install.package_source > capabilities.yaml lookup.
    package_source = package
    source_origin = "--package override"
    if package_source is None:
        package_source = existing.install.package_source or None
        if package_source:
            source_origin = "manifest.install.package_source"
    if package_source is None and capabilities_path and os.path.exists(capabilities_path):
        with open(capabilities_path) as f:
            yconfig = yaml.safe_load(f)
        for p in (yconfig or {}).get("capabilities", []):
            if p.get("name") == capability_name:
                package_source = p.get("package")
                source_origin = f"{capabilities_path} lookup"
                break
    if package_source is None:
        typer.echo(
            f"Cannot recover package spec for {capability_name!r}. Manifest predates the "
            f"`package_source` field. Supply --package <spec> or "
            f"--capabilities capabilities.yaml.",
            err=True,
        )
        raise typer.Exit(code=1)
    
    # Snapshot the original installed_at so we can preserve it after _generate_manifest
    # rewrites the file (which would otherwise set installed_at = regenerated_at = now).
    original_installed_at = existing.install.installed_at
    
    typer.echo(f"Regenerating {capability_name} from {package_source} ({source_origin})...")
    out_path = _generate_manifest(env_name, package_source, cfg.manifests_dir)
    if out_path is None:
        typer.echo(f"Regeneration failed for {capability_name}", err=True)
        raise typer.Exit(code=1)
    
    # Post-write fix-up: restore installed_at from the pre-existing manifest.
    # _generate_manifest sets both installed_at + regenerated_at to "now"; for
    # regenerations the install timestamp should be the original install moment.
    # If the original manifest had no installed_at (very old legacy), let
    # _generate_manifest's "now" stand — better than empty.
    if original_installed_at:
        try:
            refreshed = load_manifest(out_path)
            refreshed.install.installed_at = original_installed_at
            write_manifest(out_path, refreshed)
        except (json.JSONDecodeError, ValueError, IOError) as e:
            typer.echo(f"Warning: failed to restore installed_at on {out_path}: {e}")
    
    typer.echo(f"✓ Regenerated manifest at {out_path}")

In [ ]:
#| export
def _generate_adapter_manifest(
    env_name: str,        # Conda env containing the adapter impl (the tool's worker env)
    target: str,          # Adapter impl spec 'module:ClassName'
    manifests_dir: Path,  # Manifest output directory
) -> Path:  # The written adapter-manifest path
    """Introspect a task-adapter impl in-env and write its adapter manifest (CR-17 pt 2).

    The non-typer core shared by the `generate-adapter-manifest` command and
    `install-all`'s per-capability `adapters:` entries (stage 6 J10 -- the I6/J8
    install-pipeline gap: adapter installation + registration ride the SAME
    pipeline as capability installation, never manual afterthoughts).
    Raises ValueError on a malformed target and CalledProcessError when the
    in-env introspection fails; callers decide the exit posture.
    """
    if ":" not in target:
        raise ValueError(f"adapter target must be 'module:ClassName', got {target!r}")
    module_name, class_name = target.rsplit(":", 1)
    introspection_script = f"""
import inspect, json
import importlib

mod = importlib.import_module("{module_name}")
cls = getattr(mod, "{class_name}")
task_name = getattr(cls, "task_name", "") or ""
proto = getattr(cls, "required_tool_protocol", None)
if not task_name:
    raise SystemExit("[adapter-introspection] class has no task_name")
if proto is None:
    raise SystemExit("[adapter-introspection] class has no required_tool_protocol "
                     "(a provisional None protocol is not registrable)")

methods = []
properties = []
for name in sorted(dir(proto)):
    if name.startswith("_"):
        continue
    attr = inspect.getattr_static(proto, name)
    if isinstance(attr, property):
        properties.append(name)
        continue
    if isinstance(attr, (staticmethod, classmethod)):
        attr = attr.__func__
    if inspect.isfunction(attr):
        try:
            sig = inspect.signature(attr)
            params = [p for p in sig.parameters if p != "self"]
            sig_str = str(sig)
        except (ValueError, TypeError):
            params, sig_str = [], "(...)"
        methods.append({{"name": name, "signature": sig_str, "params": params}})

root = "{module_name}".split(".")[0]
try:
    version = getattr(importlib.import_module(root), "__version__", "0.0.0")
except Exception:
    version = "0.0.0"

print(json.dumps({{
    "unit": "adapter",
    "name": "{module_name}.{class_name}",
    "version": version,
    "task_name": task_name,
    "module": "{module_name}",
    "class": "{class_name}",
    "required_tool_protocol": proto.__module__ + "." + proto.__qualname__,
    "protocol_members": {{"methods": methods, "properties": properties}},
}}, indent=2))
"""
    introspection_fd, introspection_path = tempfile.mkstemp(
        prefix=f"cjm_adapter_introspect_{env_name}_", suffix=".py")
    try:
        with os.fdopen(introspection_fd, "w") as f:
            f.write(introspection_script)
        conda_cmd = _get_conda_cmd_str()
        cmd = f'{conda_cmd} run -n {env_name} python "{introspection_path}"'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True)
        out = result.stdout.strip()
        start, end = out.find("{"), out.rfind("}") + 1
        data = json.loads(out[start:end])
        data["generated_at"] = datetime.now(timezone.utc).isoformat()
        data["conda_env"] = env_name
        manifests_dir.mkdir(parents=True, exist_ok=True)
        out_file = manifests_dir / f"adapter-{data['task_name']}-{class_name}.json"
        with open(out_file, "w") as f:
            json.dump(data, f, indent=2)
        typer.echo(f"Adapter manifest written: {out_file}")
        typer.echo(f"  task: {data['task_name']}  protocol: {data['required_tool_protocol']}")
        typer.echo(f"  methods: {len(data['protocol_members']['methods'])}  "
                   f"properties: {len(data['protocol_members']['properties'])}")
        return out_file
    finally:
        try:
            os.unlink(introspection_path)
        except OSError:
            pass

In [ ]:
#| export
@app.command("generate-adapter-manifest")
def generate_adapter_manifest(
    env_name: str = typer.Argument(..., help="Conda env containing the adapter impl (the tool's worker env)"),
    target: str = typer.Argument(..., help="Adapter impl spec 'module:ClassName'"),
):
    """CR-17 pt 2 (stage 4): introspect a task-adapter impl in-env and write its adapter manifest.

    The adapter manifest is the REGISTRATION unit (pass-2 Thread 3): task_name
    + required_tool_protocol members (names + parameter lists + signatures)
    recorded IN-ENV where the protocol is importable, so host-side
    compatibility matching works against UNLOADED capabilities with zero
    protocol imports host-side. Written to the same manifests dir capability
    manifests live in; `discover_manifests()` routes by the `unit` key.

    Thin typer wrapper over `_generate_adapter_manifest` (stage 6 J10:
    install-all runs the same core for per-capability `adapters:` entries).
    """
    cfg = get_config()
    try:
        _generate_adapter_manifest(env_name, target, cfg.manifests_dir)
    except ValueError as e:
        typer.echo(str(e), err=True)
        raise typer.Exit(1)
    except subprocess.CalledProcessError as e:
        typer.echo(f"Adapter introspection failed in env {env_name}:", err=True)
        typer.echo(e.stderr or e.stdout or "", err=True)
        raise typer.Exit(1)

In [ ]:
#| export
def _conda_env_exists_configured(
    env_name: str  # Name of the conda environment
) -> bool:  # True if environment exists
    """Check if conda environment exists using configured conda command."""
    cfg = get_config()
    cmd_parts = get_conda_command(cfg)
    # Use the first part as the conda command for conda_env_exists
    conda_cmd = cmd_parts[0]
    
    # For micromamba with root prefix, we need to filter envs by the prefix path
    if cfg.runtime.conda_type == CondaType.MICROMAMBA and cfg.runtime.prefix:
        # micromamba -r <prefix> env list --json returns ALL known envs,
        # so we need to filter by the root prefix path
        try:
            result = subprocess.run(
                cmd_parts + ["env", "list", "--json"],
                capture_output=True,
                text=True
            )
            if result.returncode != 0:
                return False
            
            data = json.loads(result.stdout)
            # Get the absolute prefix path for comparison
            prefix_str = str(cfg.runtime.prefix.resolve())
            
            for path in data.get('envs', []):
                # Only consider envs under our root prefix
                if path.startswith(prefix_str):
                    if Path(path).name == env_name:
                        return True
            return False
        except (subprocess.SubprocessError, json.JSONDecodeError, FileNotFoundError):
            return False
    elif cfg.runtime.conda_type == CondaType.MICROMAMBA:
        # System micromamba without root prefix
        return conda_env_exists(env_name, conda_cmd)
    else:
        return conda_env_exists(env_name, conda_cmd)

In [ ]:
#| export
@app.command()
def install_all(
    capabilities_path:Optional[str]=typer.Option(None, "--capabilities", help="Path to capabilities.yaml (default: cjm.yaml capabilities_config)"),
    substrate_source:str=typer.Option("cjm-substrate", "--substrate-source", help="Substrate package spec installed into every worker env (default: published; pass a path or '-e <path>' for local-editable dev)"),
    force:bool=typer.Option(False, help="Force recreation of environments")
) -> None:
    """Install and register all capabilities defined in capabilities.yaml.

    Per-capability `adapters:` entries ride the same pipeline (stage 6 J10; closes
    the I6/J8 manual-step gap): each entry's `lib` is pip-installed into the
    worker env alongside the interface libs, and each `impl`
    ('module:ClassName') gets its adapter manifest generated right after the
    capability manifest -- INSTALL puts code in envs, REGISTRATION is per-unit
    manifests (pass-2 Thread 3), one command does both.
    """
    cfg = get_config()
    
    # Schema v2 (PILLAR 2): default the capabilities file from cjm.yaml's
    # capabilities_config (parent-walk-discovered) so install-all runs flagless.
    if capabilities_path is None:
        capabilities_path = str(cfg.capabilities_config)

    # Check runtime availability
    _check_runtime_available()
    
    if not os.path.exists(capabilities_path):
        typer.echo(f"Capabilities file not found: {capabilities_path}", err=True)
        raise typer.Exit(code=1)

    with open(capabilities_path) as f:
        config = yaml.safe_load(f)

    # Setup manifest directory using config
    manifest_dir = cfg.manifests_dir
    manifest_dir.mkdir(parents=True, exist_ok=True)

    capabilities = config.get('capabilities', [])
    print(f"Found {len(capabilities)} capabilities to process.")
    
    # Get the conda command string for shell commands
    conda_cmd = _get_conda_cmd_str()
    
    # Determine if using micromamba (different command syntax)
    is_micromamba = cfg.runtime.conda_type == CondaType.MICROMAMBA
    
    # Track temp files to clean up
    temp_files_to_cleanup: List[Path] = []

    try:
        for capability in capabilities:
            name = capability.get('name')
            env_name = capability.get('env_name')
            print(f"\n=== Processing {name} ({env_name}) ===")

            # 1. Check if Env Exists (using configured conda command)
            env_exists = _conda_env_exists_configured(env_name)

            # 2. Create Environment
            if not env_exists or force:
                # SAFETY: If forcing, remove old one first to avoid flag issues
                if env_exists and force:
                    print(f"Removing existing environment {env_name}...")
                    run_cmd(f"{conda_cmd} env remove -n {env_name} -y")

                if 'env_file' in capability:
                    # Resolve env_file (download if URL)
                    local_env_file, temp_file = _resolve_env_file(capability['env_file'])
                    if temp_file:
                        temp_files_to_cleanup.append(temp_file)
                    
                    # micromamba uses 'create' directly, conda uses 'env create'
                    if is_micromamba:
                        run_cmd(f"{conda_cmd} create -f {local_env_file} -n {env_name} -y")
                    else:
                        run_cmd(f"{conda_cmd} env create -f {local_env_file} -n {env_name}")
                elif 'python_version' in capability:
                    run_cmd(f"{conda_cmd} create -n {env_name} python={capability['python_version']} -y")

            # 3. Install Dependencies (Pip)
            base_pip_cmd = f"{conda_cmd} run -n {env_name} pip install"

            # Schema v2 (PILLAR 2): the substrate is AUTO-INJECTED into every
            # worker env (was carried implicitly via each capability's interface_libs).
            # Default = the published package; --substrate-source overrides for
            # local-editable dev.
            run_cmd(f"{base_pip_cmd} {substrate_source}")
            
            if 'interface_libs' in capability:
                libs = " ".join(capability['interface_libs'])
                run_cmd(f"{base_pip_cmd} {libs}")

            # Adapter impl libraries ride the same env install (J10)
            adapter_entries = capability.get('adapters') or []
            adapter_libs = [a['lib'] for a in adapter_entries
                            if isinstance(a, dict) and a.get('lib')]
            if adapter_libs:
                run_cmd(f"{base_pip_cmd} {' '.join(adapter_libs)}")

            if 'package' in capability:
                run_cmd(f"{base_pip_cmd} {capability['package']}")

            # 4. Generate Manifest.
            # CR-8: _generate_manifest now writes install.conda_env directly
            # (it has env_name as a parameter), so the legacy post-write
            # conda_env step is no longer needed.
            pkg_source = capability['package']
            _generate_manifest(env_name, pkg_source, manifest_dir)

            # 5. Generate adapter manifests (registration units) for any
            # adapter impls this worker env carries. Failure is LOUD -- a
            # half-registered adapter is the silent-inert pattern (G11/I6).
            for entry in adapter_entries:
                impl = entry.get('impl') if isinstance(entry, dict) else None
                if not impl:
                    typer.echo(f"{name}: adapters entry missing 'impl' "
                               f"('module:ClassName'): {entry!r}", err=True)
                    raise typer.Exit(code=1)
                try:
                    _generate_adapter_manifest(env_name, impl, manifest_dir)
                except (ValueError, subprocess.CalledProcessError) as e:
                    detail = ""
                    if isinstance(e, subprocess.CalledProcessError):
                        detail = (e.stderr or e.stdout or "").strip()
                    typer.echo(f"Adapter manifest generation FAILED for {impl} "
                               f"in {env_name}: {detail or e}", err=True)
                    raise typer.Exit(code=1)

        print("\n All operations complete.")
    
    finally:
        # Clean up any temporary files
        for temp_file in temp_files_to_cleanup:
            try:
                temp_file.unlink()
            except IOError:
                pass

## Setup Host Environment

The `setup-host` command prepares the host application's Python environment by installing all unique interface libraries referenced in `capabilities.yaml`. This is separate from `install-all` which sets up isolated capability environments.

In [ ]:
#| export
@app.command("setup-host")
def setup_host(
    capabilities_path:str=typer.Option("capabilities.yaml", "--capabilities", help="Path to capabilities.yaml file"),
    yes:bool=typer.Option(False, "--yes", "-y", help="Skip confirmation prompt")
) -> None:
    """Install interface libraries in the current Python environment."""
    if not os.path.exists(capabilities_path):
        typer.echo(f"Capabilities file not found: {capabilities_path}", err=True)
        raise typer.Exit(code=1)

    with open(capabilities_path) as f:
        config = yaml.safe_load(f)

    # Collect unique interface libraries from all capabilities
    capabilities = config.get('capabilities', [])
    all_libs: set[str] = set()
    
    for capability in capabilities:
        interface_libs = capability.get('interface_libs', [])
        all_libs.update(interface_libs)
    
    # Filter out cjm-substrate (already installed since we're running this CLI)
    all_libs = {lib for lib in all_libs 
                if 'cjm-substrate.git' not in lib and lib != 'cjm-substrate'}
    
    if not all_libs:
        typer.echo("No interface libraries found in config.")
        raise typer.Exit(code=0)

    # Display what will be installed
    typer.echo(f"Reading {capabilities_path}...")
    typer.echo(f"Found {len(all_libs)} unique interface libraries:")
    for lib in sorted(all_libs):
        typer.echo(f"  - {lib}")
    typer.echo("")

    # Confirm with user unless --yes flag provided
    if not yes:
        confirm = typer.confirm("Install these in the current Python environment?")
        if not confirm:
            typer.echo("Aborted.")
            raise typer.Exit(code=0)

    # Install libraries using the current Python interpreter
    libs_str = " ".join(sorted(all_libs))
    run_cmd(f"{sys.executable} -m pip install {libs_str}")
    
    typer.echo("\nHost environment setup complete.")

## Estimate Disk Space

The `estimate-size` command estimates the disk space required for capability environments before installation. It uses conda's dry-run feature for accurate conda package sizes and queries PyPI for pip package sizes.

In [ ]:
#| export
_PYPI_404_CACHE: set[str] = set()

In [ ]:
#| export
def _format_size(
    size_bytes: int  # Size in bytes
) -> str:  # Human-readable size string
    """Format bytes as human-readable string."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024:
            return f"{size_bytes:.1f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.1f} PB"

In [ ]:
#| export
def _get_pypi_size(
    package_spec: str  # Package name or git URL
) -> tuple[int, str]:  # (size_bytes, package_name)
    """Query PyPI for package download size.
    
    SG-37: PyPI normalizes package names to dash-form per PEP 503, so we try
    the dash form first and fall back to the underscore form. Without this,
    cjm-* packages (whose repo names contain dashes) silently 404 because
    the old heuristic only tried the underscored form.
    """
    # Extract a raw repo-name fragment from various spec formats.
    raw_name = package_spec
    if 'github.com' in package_spec and '.git' in package_spec:
        raw_name = package_spec.split('/')[-1].replace('.git', '')
    elif package_spec.startswith('git+'):
        raw_name = package_spec.rstrip('/').split('/')[-1]
    elif '[' in package_spec:
        raw_name = package_spec.split('[')[0]
    elif '>=' in package_spec:
        raw_name = package_spec.split('>=')[0]
    elif '==' in package_spec:
        raw_name = package_spec.split('==')[0]
    
    # Build ordered candidate list: dash-form first (canonical per PEP 503),
    # then underscore-form as fallback. Dedupe while preserving order.
    dash_form = raw_name.replace('_', '-')
    underscore_form = raw_name.replace('-', '_')
    candidates: List[str] = []
    for cand in (dash_form, underscore_form):
        if cand and cand not in candidates:
            candidates.append(cand)
    
    for package_name in candidates:
        if package_name in _PYPI_404_CACHE:
            continue
        try:
            url = f"https://pypi.org/pypi/{package_name}/json"
            with urllib.request.urlopen(url, timeout=10) as resp:
                data = json.loads(resp.read().decode())
            
            # Get the largest file size (usually the wheel)
            urls = data.get('urls', [])
            if urls:
                max_size = max(u.get('size', 0) for u in urls)
                return max_size, package_name
            return 0, package_name
        except urllib.error.HTTPError as e:
            if e.code == 404:
                _PYPI_404_CACHE.add(package_name)
                continue
        except (urllib.error.URLError, json.JSONDecodeError):
            pass
    
    # Nothing matched; surface the canonical form to the caller for display.
    return 0, candidates[0] if candidates else raw_name

In [ ]:
#| export
def _estimate_conda_size(
    env_file: str,  # Path or URL to environment.yml
    env_name: str  # Target environment name
) -> tuple[int, int]:  # (total_bytes, package_count)
    """Estimate conda package sizes using dry-run."""
    cfg = get_config()
    
    # Resolve env_file (download if URL)
    temp_file = None
    try:
        local_env_file, temp_file = _resolve_env_file(env_file)
    except RuntimeError:
        return 0, 0
    
    try:
        cmd_parts = build_conda_command(cfg, "create", "-f", local_env_file, "-n", env_name, "--dry-run", "--json")
        
        # For micromamba, the command is slightly different
        if cfg.runtime.conda_type != CondaType.MICROMAMBA:
            # conda/mamba use "env create" for environment files
            cmd_parts = build_conda_command(cfg, "env", "create", "-f", local_env_file, "-n", env_name, "--dry-run", "--json")
        
        result = subprocess.run(
            cmd_parts,
            capture_output=True, text=True
        )
        
        if result.returncode != 0:
            return 0, 0
        
        data = json.loads(result.stdout)
        fetch_actions = data.get('actions', {}).get('FETCH', [])
        total_size = sum(pkg.get('size', 0) for pkg in fetch_actions)
        return total_size, len(fetch_actions)
        
    except (subprocess.SubprocessError, json.JSONDecodeError):
        return 0, 0
    finally:
        # Clean up temp file if we created one
        if temp_file:
            try:
                temp_file.unlink()
            except IOError:
                pass

In [ ]:
#| export
def _estimate_pip_sizes(
    packages: list[str]  # List of pip package specs
) -> tuple[int, int, list[tuple[str, int]]]:  # (total_bytes, found_count, [(name, size), ...])
    """Estimate pip package sizes from PyPI."""
    total_size = 0
    found_count = 0
    details = []
    
    for pkg in packages:
        size, name = _get_pypi_size(pkg)
        details.append((name, size))
        if size > 0:
            total_size += size
            found_count += 1
    
    return total_size, found_count, details

## List Capabilities

The `list` command shows installed capabilities by scanning manifest files in the configured manifests directory (defaults to `~/.cjm/manifests/`). It can optionally cross-reference with a capabilities.yaml file and check conda environment status.

In [ ]:
#| export
def _get_conda_envs() -> set[str]: # Set of existing conda environment names
    """Get list of existing conda environment names using configured conda command."""
    cfg = get_config()
    cmd_parts = build_conda_command(cfg, "env", "list", "--json")
    
    try:
        result = subprocess.run(
            cmd_parts,
            capture_output=True, text=True
        )
        if result.returncode == 0:
            data = json.loads(result.stdout)
            envs = set()
            
            # For micromamba with root prefix, filter by the prefix path
            if cfg.runtime.conda_type == CondaType.MICROMAMBA and cfg.runtime.prefix:
                prefix_str = str(cfg.runtime.prefix.resolve())
                for path in data.get('envs', []):
                    if path.startswith(prefix_str):
                        env_name = Path(path).name
                        envs.add(env_name)
            else:
                # For conda/mamba or system micromamba, use all envs
                for path in data.get('envs', []):
                    env_name = Path(path).name
                    envs.add(env_name)
            
            return envs
    except (subprocess.SubprocessError, json.JSONDecodeError):
        pass
    return set()

In [ ]:
#| export
def _v2_to_legacy_flat_view(
    raw: Dict[str, Any]  # Manifest JSON dict as read from disk
) -> Dict[str, Any]:  # Flat-shaped dict (install + code merged at top level)
    """REMOVE-AFTER-OVERHAUL: produce a legacy flat-shaped view of a manifest.
    
    Consumer code in `list_capabilities` + `remove_capability` still reads manifests as
    flat dicts (`manifest.get('python_path')`, etc.). When the on-disk manifest
    is v2.0 nested, we flatten install + code sections to the top level so
    those consumers don't need migration in the same PR. The shim retires when
    consumers migrate to `load_manifest` for typed access.
    
    v1.0 manifests pass through unchanged.
    """
    if not isinstance(raw, dict) or raw.get("format_version") != CURRENT_FORMAT_VERSION:
        return raw
    install = raw.get("install", {}) or {}
    code = raw.get("code", {}) or {}
    flat: Dict[str, Any] = {}
    flat.update(install)
    flat.update(code)
    flat["format_version"] = raw.get("format_version", CURRENT_FORMAT_VERSION)
    return flat

In [ ]:
#| export
def _get_installed_manifests(
    manifest_dir:Optional[Path]=None # Directory to scan (uses config default if None)
) -> list[dict]: # List of manifest dictionaries
    """Load all manifest JSON files from the manifest directory.
    
    Returns flat-shaped dicts regardless of on-disk format (v2.0 manifests are
    flattened via `_v2_to_legacy_flat_view`). Consumers that want typed access
    should use `load_manifest` from `cjm_substrate.core.manifest_format`
    instead.
    """
    if manifest_dir is None:
        manifest_dir = get_config().manifests_dir
    
    manifests = []
    
    if not manifest_dir.exists():
        return manifests
    
    for manifest_file in manifest_dir.glob("*.json"):
        try:
            with open(manifest_file) as f:
                raw = json.load(f)
            manifest = _v2_to_legacy_flat_view(raw)
            manifest['_manifest_path'] = str(manifest_file)
            manifests.append(manifest)
        except (json.JSONDecodeError, IOError):
            pass
    
    return manifests

In [ ]:
#| export
def _extract_env_from_python_path(
    python_path:str # Path like /home/user/miniforge3/envs/my-env/bin/python
) -> str: # Extracted environment name or empty string
    """Extract conda environment name from python_path."""
    if not python_path:
        return ''
    
    # Look for /envs/<env_name>/ pattern
    path_parts = Path(python_path).parts
    for i, part in enumerate(path_parts):
        if part == 'envs' and i + 1 < len(path_parts):
            return path_parts[i + 1]
    
    return ''

In [ ]:
#| export
@app.command("list")
def list_capabilities(
    capabilities_path:Optional[str]=typer.Option(None, "--capabilities", help="Path to capabilities.yaml for cross-reference"),
    show_envs:bool=typer.Option(False, "--envs", "-e", help="Show conda environment status")
) -> None:
    """List installed capabilities from manifest directory."""
    cfg = get_config()
    manifests = _get_installed_manifests()
    
    if not manifests:
        typer.echo(f"No capabilities found in {cfg.manifests_dir}")
        raise typer.Exit(code=0)
    
    # Load config for cross-reference if provided
    config_capabilities = {}
    if capabilities_path and os.path.exists(capabilities_path):
        with open(capabilities_path) as f:
            config = yaml.safe_load(f)
        for p in config.get('capabilities', []):
            config_capabilities[p.get('name')] = p
    
    # Get conda envs if requested
    conda_envs = _get_conda_envs() if show_envs else set()
    
    typer.echo(f"=== Installed Capabilities ({len(manifests)}) ===\n")
    
    for manifest in sorted(manifests, key=lambda m: m.get('name', '')):
        name = manifest.get('name', 'unknown')
        version = manifest.get('version', '?')
        
        # Get env name from manifest or extract from python_path
        env_name = manifest.get('conda_env', '')
        if not env_name:
            env_name = _extract_env_from_python_path(manifest.get('python_path', ''))
        
        # Basic info line
        typer.echo(f"{name} v{version}")
        
        # Environment status
        if show_envs and env_name:
            env_status = "exists" if env_name in conda_envs else "missing"
            typer.echo(f"  Env: {env_name} ({env_status})")
        elif show_envs:
            typer.echo(f"  Env: (not specified in manifest)")
        
        # Cross-reference with config
        if capabilities_path:
            if name in config_capabilities:
                cfg_env = config_capabilities[name].get('env_name', '')
                if cfg_env and cfg_env != env_name:
                    typer.echo(f"  Config env: {cfg_env} (differs from manifest)")
            else:
                typer.echo(f"  (not in {capabilities_path})")
        
        typer.echo("")

## Observability commands (CR-14 follow-up)

`cjm-ctl logs` is the operator projection over the two stores (the
replacement for the retired `tail .cjm/logs/<capability>.log` habit): the
default view is structured diagnostics records (worker `logger.*` output,
job-stamped); `--chunks` shows the raw stream pump (the death-rattle floor);
`--journal` shows the account-of-action. `--follow` polls the `seq` cursor —
the same exact-cursor mechanism that replaced `LOG_APPENDED`.

`cjm-ctl retention` applies the diagnostics retention policy on demand
(the startup sweep in `CapabilityManager` is the automatic invocation). The
journal has no retention surface by design.

In [ ]:
#| export
def _fmt_short(value: Optional[str], width: int = 8) -> str:
    """First `width` chars of an id, or '-' for None (display only)."""
    return (value or "-")[:width]


def _compact_payload(payload: Dict[str, Any], max_len: int = 160) -> str:
    """One-line payload rendering; the bulky job_snapshot collapses to its status."""
    d = dict(payload or {})
    snap = d.get("job_snapshot")
    if isinstance(snap, dict):
        d["job_snapshot"] = f"<snapshot:{snap.get('status', '?')}>"
    s = json.dumps(d, default=str, sort_keys=True)
    return s if len(s) <= max_len else s[: max_len - 1] + "…"


@app.command("logs")
def logs_command(
    job:Annotated[Optional[str], typer.Option("--job", help="Filter to one queue job id (exact)")]=None,
    run:Annotated[Optional[str], typer.Option("--run", help="Filter to one host run id (implies --journal)")]=None,
    session:Annotated[Optional[str], typer.Option("--session", help="Filter to one worker session id")]=None,
    level:Annotated[Optional[str], typer.Option("--level", help="Diagnostics level filter (e.g. WARNING)")]=None,
    journal:Annotated[bool, typer.Option("--journal", help="Show the journal (account-of-action) instead of diagnostics records")]=False,
    chunks:Annotated[bool, typer.Option("--chunks", help="Show raw stream chunks (death-rattle floor)")]=False,
    limit:Annotated[int, typer.Option("--limit", "-n", help="Most recent N entries (0 = all)")]=50,
    follow:Annotated[bool, typer.Option("--follow", "-f", help="Poll for new entries (Ctrl-C to stop)")]=False,
) -> None:
    """Tail / follow the observability stores (CR-14).

    Default view: structured diagnostics records (worker logger output,
    EXACTLY job-stamped via the call envelope). `--chunks`: the raw stream
    pump. `--journal`: the durable account-of-action (job lifecycle, worker
    spawn/death, admission, config, runs, worker-reported accounts).
    `--follow` polls the store's seq cursor — exact, no byte offsets.
    """
    from cjm_substrate.core.diagnostics_store import LocalDiagnosticsStore
    from cjm_substrate.core.journal_store import LocalJournalStore

    if run and not journal:
        journal = True  # run correlation lives on journal rows
    if journal and chunks:
        typer.echo("--journal and --chunks are different views; pick one.", err=True)
        raise typer.Exit(code=1)

    cfg = get_config()
    lim = None if limit == 0 else limit

    if journal:
        store = LocalJournalStore(cfg.journal_db_path)
        db = cfg.journal_db_path

        def fetch(after_seq=None, lim_=lim):
            return store.query(job_id=job, run_id=run, worker_session_id=session,
                               after_seq=after_seq, limit=lim_,
                               descending=(after_seq is None))

        def render(ev):
            line = (f"{ev.ts:%Y-%m-%d %H:%M:%S} {ev.event_type:<22} "
                    f"[job={_fmt_short(ev.job_id)} run={ev.run_id or '-'}] "
                    f"{ev.capability_instance_id or ev.capability_name or '-'}"
                    f"{' (worker)' if ev.worker_reported else ''} "
                    f"{_compact_payload(ev.payload)}")
            return line
    elif chunks:
        dstore = LocalDiagnosticsStore(cfg.diagnostics_db_path)
        db = cfg.diagnostics_db_path

        def fetch(after_seq=None, lim_=lim):
            return dstore.query_chunks(worker_session_id=session,
                                       after_seq=after_seq, limit=lim_,
                                       descending=(after_seq is None))

        def render(c):
            return f"{c.ts:%Y-%m-%d %H:%M:%S} [{_fmt_short(c.worker_session_id)}] {c.content}"
    else:
        dstore = LocalDiagnosticsStore(cfg.diagnostics_db_path)
        db = cfg.diagnostics_db_path

        def fetch(after_seq=None, lim_=lim):
            return dstore.query_records(job_id=job, worker_session_id=session,
                                        level=level.upper() if level else None,
                                        after_seq=after_seq, limit=lim_,
                                        descending=(after_seq is None))

        def render(r):
            line = (f"{r.ts:%Y-%m-%d %H:%M:%S} {r.level:<8} "
                    f"[{_fmt_short(r.worker_session_id)}|{_fmt_short(r.job_id)}] "
                    f"{r.logger_name}: {r.message}")
            if r.exc_text:
                line += "\n" + "\n".join("    " + l for l in r.exc_text.splitlines())
            return line

    if not db.exists():
        typer.echo(f"No store at {db} (nothing has run here yet).")
        raise typer.Exit(code=0)

    rows = fetch()
    rows = list(reversed(rows))  # newest-first fetch -> chronological display
    for row in rows:
        typer.echo(render(row))
    if not follow:
        return

    import time as _time
    cursor = max((r.seq or 0) for r in rows) if rows else 0
    typer.echo(f"-- following {db.name} (Ctrl-C to stop) --", err=True)
    try:
        while True:
            _time.sleep(1.0)
            new = fetch(after_seq=cursor, lim_=None)
            for row in new:
                typer.echo(render(row))
                cursor = max(cursor, row.seq or 0)
    except KeyboardInterrupt:
        pass

In [ ]:
#| export
@app.command("retention")
def retention_command(
    max_age_days:Annotated[Optional[float], typer.Option(
        "--max-age-days", help="Delete diagnostics rows older than this (overrides cjm.yaml)")]=None,
    max_total_mb:Annotated[Optional[float], typer.Option(
        "--max-total-mb", help="Delete oldest rows until diagnostics.db is under this budget")]=None,
) -> None:
    """Apply the diagnostics retention policy now (CR-14).

    The explicit half of the invocation policy (CapabilityManager's startup
    sweep is the automatic half). Defaults come from `cjm.yaml`'s
    `substrate.diagnostics_retention_days` / `diagnostics_retention_max_mb`.
    The JOURNAL is never touched — it has no retention surface by design.
    """
    from cjm_substrate.core.diagnostics_store import LocalDiagnosticsStore

    cfg = get_config()
    sub = cfg.substrate
    days = max_age_days if max_age_days is not None else float(
        getattr(sub, "diagnostics_retention_days", 0.0) or 0.0)
    budget = max_total_mb if max_total_mb is not None else getattr(
        sub, "diagnostics_retention_max_mb", None)
    if days <= 0 and budget is None:
        typer.echo("Retention disabled (diagnostics_retention_days <= 0, no "
                   "size budget). Pass --max-age-days / --max-total-mb or set "
                   "the cjm.yaml substrate policy.")
        raise typer.Exit(code=0)
    db = cfg.diagnostics_db_path
    if not db.exists():
        typer.echo(f"No diagnostics store at {db}; nothing to do.")
        raise typer.Exit(code=0)

    store = LocalDiagnosticsStore(db)
    before_mb = db.stat().st_size / (1024 * 1024)
    deleted = store.apply_retention(
        max_age_days=days if days > 0 else None, max_total_mb=budget)
    typer.echo(
        f"diagnostics retention: deleted {deleted['records_deleted']} record(s) "
        f"+ {deleted['chunks_deleted']} chunk(s) "
        f"(db {before_mb:.1f} MB; freed pages reuse before the file shrinks). "
        f"Journal untouched.")

## Remove Capability

The `remove` command removes a capability's manifest and optionally its conda environment. It can look up the environment name from the manifest or a config file.

In [ ]:
#| export
@app.command("remove")
def remove_capability(
    capability_name:str=typer.Argument(..., help="Name of the capability to remove"),
    capabilities_path:Optional[str]=typer.Option(None, "--capabilities", help="Path to capabilities.yaml for env name lookup"),
    keep_env:bool=typer.Option(False, "--keep-env", help="Keep the conda environment, only remove manifest"),
    yes:bool=typer.Option(False, "--yes", "-y", help="Skip confirmation prompt")
) -> None:
    """Remove a capability's manifest and conda environment."""
    cfg = get_config()
    manifest_dir = cfg.manifests_dir
    manifest_path = manifest_dir / f"{capability_name}.json"
    
    # Find the manifest. CR-8: read via the legacy-flat-view shim so v2.0
    # nested manifests work with the existing dict-access code below.
    manifest = None
    if manifest_path.exists():
        try:
            with open(manifest_path) as f:
                raw = json.load(f)
            manifest = _v2_to_legacy_flat_view(raw)
        except (json.JSONDecodeError, IOError):
            pass
    
    if not manifest:
        typer.echo(f"Capability manifest not found: {capability_name}", err=True)
        typer.echo(f"  Expected at: {manifest_path}")
        raise typer.Exit(code=1)
    
    # Determine conda environment name (try multiple sources)
    env_name = manifest.get('conda_env', '')
    
    # Fallback 1: Extract from python_path in manifest
    if not env_name:
        env_name = _extract_env_from_python_path(manifest.get('python_path', ''))
    
    # Fallback 2: Check capabilities file
    if not env_name and capabilities_path and os.path.exists(capabilities_path):
        with open(capabilities_path) as f:
            config = yaml.safe_load(f)
        for p in config.get('capabilities', []):
            if p.get('name') == capability_name:
                env_name = p.get('env_name', '')
                break
    
    # Check if env exists
    conda_envs = _get_conda_envs()
    env_exists = env_name in conda_envs if env_name else False
    
    # Show what will be removed
    typer.echo(f"=== Remove Capability: {capability_name} ===\n")
    typer.echo(f"Manifest: {manifest_path}")
    
    if env_name:
        if env_exists:
            if keep_env:
                typer.echo(f"Conda env: {env_name} (will be kept)")
            else:
                typer.echo(f"Conda env: {env_name} (will be removed)")
        else:
            typer.echo(f"Conda env: {env_name} (does not exist)")
    else:
        typer.echo(f"Conda env: (not specified)")
    
    typer.echo("")
    
    # Confirm
    if not yes:
        confirm = typer.confirm("Proceed with removal?")
        if not confirm:
            typer.echo("Aborted.")
            raise typer.Exit(code=0)
    
    # Remove conda environment using configured command
    if env_name and env_exists and not keep_env:
        typer.echo(f"Removing conda environment: {env_name}...")
        try:
            cmd_parts = build_conda_command(cfg, "env", "remove", "-n", env_name, "-y")
            subprocess.run(cmd_parts, check=True)
            typer.echo(f"  Removed: {env_name}")
        except subprocess.CalledProcessError as e:
            typer.echo(f"  Warning: Failed to remove conda env: {e}", err=True)
    
    # Remove manifest file
    typer.echo(f"Removing manifest: {manifest_path}...")
    try:
        manifest_path.unlink()
        typer.echo(f"  Removed: {manifest_path.name}")
    except IOError as e:
        typer.echo(f"  Error removing manifest: {e}", err=True)
        raise typer.Exit(code=1)
    
    typer.echo(f"\nCapability '{capability_name}' removed successfully.")

## Validate Command (SG-6)

The `validate` command checks that a manifest JSON file or a `capabilities.yaml` file
conforms to the substrate's expected structure. Auto-detects format from the
file extension (`.json` → manifest, `.yaml`/`.yml` → capabilities.yaml).

Composes with CR-8 (manifest schema authority): the validator checks the
nested v2.0 manifest layout. (The legacy flat v1.0 shape + its validator were
removed at SG-48.)

In [ ]:
#| export
def _validate_resources_block(
    res: Any,  # resources sub-dict (may be None or non-dict; we type-check here)
    path_prefix: str,  # Error message prefix
) -> List[str]:
    """Phase 5a: type-check the resources block. Shared between v1.0 and v2.0."""
    errors: List[str] = []
    if res is None:
        return errors
    if not isinstance(res, dict):
        errors.append(f"{path_prefix}: 'resources' must be an object when present")
        return errors
    if "requires_gpu" in res and not isinstance(res["requires_gpu"], bool):
        errors.append(f"{path_prefix}: 'resources.requires_gpu' must be a boolean")
    for list_key in ("platforms", "accelerators"):
        if list_key in res:
            lst = res[list_key]
            if not isinstance(lst, list):
                errors.append(f"{path_prefix}: 'resources.{list_key}' must be a list")
            elif not all(isinstance(item, str) for item in lst):
                errors.append(f"{path_prefix}: 'resources.{list_key}' must contain only strings")
    return errors

In [ ]:
#| export
def _validate_manifest_v2_dict(
    data: Dict[str, Any]  # v2.0 nested manifest dict (caller already verified format_version)
) -> List[str]:  # Empty list = valid
    """CR-8: validate the nested v2.0 manifest layout.
    
    Required sections: `install` + `code`. Optional sections: `drift_tracking`
    (substrate emits it on every fresh write but legacy-via-load_manifest
    upgrades leave it empty until the first regenerate); `overrides` (free-form
    operator overlay).
    
    Required `code.*` fields mirror v1.0's required top-level fields. Required
    `install.python_path` mirrors v1.0's required top-level `python_path`.
    """
    errors: List[str] = []
    
    # Sections must be dicts when present
    install = data.get("install")
    code = data.get("code")
    drift = data.get("drift_tracking")
    overrides = data.get("overrides")
    
    if install is None:
        errors.append("manifest: required section 'install' is missing")
        install = {}
    elif not isinstance(install, dict):
        errors.append(f"manifest: 'install' must be an object, got {type(install).__name__}")
        install = {}
    
    if code is None:
        errors.append("manifest: required section 'code' is missing")
        code = {}
    elif not isinstance(code, dict):
        errors.append(f"manifest: 'code' must be an object, got {type(code).__name__}")
        code = {}
    
    if drift is not None and not isinstance(drift, dict):
        errors.append(f"manifest: 'drift_tracking' must be an object when present, got {type(drift).__name__}")
        drift = {}
    
    if overrides is not None and not isinstance(overrides, dict):
        errors.append(f"manifest: 'overrides' must be an object when present, got {type(overrides).__name__}")
    
    # Required code.* fields (mirroring v1.0 contract)
    for key in ("name", "version", "description", "module", "class"):
        val = code.get(key) if isinstance(code, dict) else None
        if val is None or (isinstance(val, str) and not val.strip()):
            errors.append(f"manifest: required field 'code.{key}' is missing or empty")
        elif not isinstance(val, str):
            errors.append(f"manifest: 'code.{key}' must be a string, got {type(val).__name__}")
    
    # Required install.python_path (the proxy needs it to spawn the worker)
    py_path = install.get("python_path") if isinstance(install, dict) else None
    if py_path is None or py_path == "":
        errors.append("manifest: required field 'install.python_path' is missing or empty")
    elif not isinstance(py_path, str):
        errors.append(f"manifest: 'install.python_path' must be a string, got {type(py_path).__name__}")
    
    # Optional install.* fields — type-check only
    for key in ("conda_env", "db_path", "installed_at", "installer_version", "package_source"):
        if isinstance(install, dict) and key in install and not isinstance(install[key], str):
            errors.append(f"manifest: 'install.{key}' must be a string when present")
    if isinstance(install, dict) and "env_vars" in install and not isinstance(install["env_vars"], dict):
        errors.append("manifest: 'install.env_vars' must be an object when present")
    
    # Optional code.* fields
    if isinstance(code, dict) and "regenerated_at" in code and code["regenerated_at"] is not None:
        if not isinstance(code["regenerated_at"], str):
            errors.append("manifest: 'code.regenerated_at' must be a string when present")
    
    # Optional code.config_schema
    if isinstance(code, dict) and "config_schema" in code and code["config_schema"] is not None:
        cs = code["config_schema"]
        if not isinstance(cs, dict):
            errors.append("manifest: 'code.config_schema' must be an object when present")
        elif "properties" in cs and not isinstance(cs["properties"], dict):
            errors.append("manifest: 'code.config_schema.properties' must be an object when present")
    
    # Optional code.resources block — type-check
    if isinstance(code, dict):
        errors.extend(_validate_resources_block(
            code.get("resources"),
            path_prefix="manifest: code",
        ))
    
    # T23: worker-env default templating — every EnvVarSpec.default's ${...}
    # placeholders must be in the allowed vocabulary, else the substrate can't
    # resolve them when it spawns the worker. Surface the capability-author bug at
    # validate/release time rather than first load_capability.
    if isinstance(code, dict) and code.get("worker_env") is not None:
        from cjm_substrate.core.capability import template_check_placeholders
        from cjm_substrate.core.errors import CapabilityConfigError
        worker_env = code["worker_env"]
        if not isinstance(worker_env, list):
            errors.append("manifest: 'code.worker_env' must be a list when present")
        else:
            for j, spec in enumerate(worker_env):
                if not isinstance(spec, dict):
                    errors.append(f"manifest: 'code.worker_env[{j}]' must be an object")
                    continue
                default = spec.get("default")
                if isinstance(default, str) and default:
                    try:
                        template_check_placeholders(default)
                    except CapabilityConfigError as e:
                        errors.append(
                            f"manifest: 'code.worker_env[{j}]' (name={spec.get('name')!r}): {e}"
                        )

    # drift_tracking.config_schema_hash type-check (optional even when present)
    if isinstance(drift, dict) and "config_schema_hash" in drift and drift["config_schema_hash"] is not None:
        if not isinstance(drift["config_schema_hash"], str):
            errors.append("manifest: 'drift_tracking.config_schema_hash' must be a string when present")
    
    return errors

In [ ]:
#| export
def _validate_manifest_dict(
    data: Any  # Loaded manifest JSON
) -> List[str]:  # List of human-readable error messages (empty == valid)
    """SG-6 + CR-8: structural validation, dispatching on `format_version`.
    
    `format_version == "2.0"` validates the nested v2.0 layout. Any other
    value (including a missing field — the legacy v1.0 flat shim was removed
    at SG-48) rejects with a single error so unknown formats fail loud rather
    than silently degrading.
    """
    if not isinstance(data, dict):
        return [f"manifest must be a JSON object, got {type(data).__name__}"]
    
    fmt = data.get("format_version")
    if fmt == "2.0":
        return _validate_manifest_v2_dict(data)
    return [
        f"manifest: unrecognized format_version {fmt!r}; expected '2.0'"
    ]

In [ ]:
#| export
def _validate_capabilities_yaml_dict(
    data: Any  # Loaded capabilities.yaml content
) -> List[str]:  # List of human-readable error messages (empty == valid)
    """Structural validation of a capabilities.yaml file.
    
    Each capability entry must have name + env_name + package, plus either env_file
    or python_version (one defines how the conda env is created).
    """
    errors: List[str] = []
    if not isinstance(data, dict):
        return [f"capabilities.yaml must be a YAML mapping at the top level, got {type(data).__name__}"]
    
    capabilities = data.get("capabilities")
    if capabilities is None:
        return ["capabilities.yaml: top-level key 'capabilities' is missing"]
    if not isinstance(capabilities, list):
        return [f"capabilities.yaml: 'capabilities' must be a list, got {type(capabilities).__name__}"]
    
    seen_names: set[str] = set()
    for i, capability in enumerate(capabilities):
        prefix = f"capabilities.yaml: capabilities[{i}]"
        if not isinstance(capability, dict):
            errors.append(f"{prefix}: must be a mapping, got {type(capability).__name__}")
            continue
        
        # Required string fields
        for key in ("name", "env_name", "package"):
            val = capability.get(key)
            if val is None or val == "":
                errors.append(f"{prefix}: required field {key!r} is missing or empty")
            elif not isinstance(val, str):
                errors.append(f"{prefix}: field {key!r} must be a string")
        
        # Name uniqueness
        name = capability.get("name")
        if isinstance(name, str) and name:
            if name in seen_names:
                errors.append(f"{prefix}: duplicate capability name {name!r}")
            seen_names.add(name)
        
        # Env creation: env_file OR python_version (at least one)
        if "env_file" not in capability and "python_version" not in capability:
            errors.append(
                f"{prefix}: must declare either 'env_file' or 'python_version' "
                f"to specify how the conda env is created"
            )
        
        if "interface_libs" in capability and not isinstance(capability["interface_libs"], list):
            errors.append(f"{prefix}: 'interface_libs' must be a list when present")

        # Adapter entries (stage 6 J10): each names the impl to register and,
        # optionally, the library that provides it (pip-installed into the env)
        if "adapters" in capability:
            adapters = capability["adapters"]
            if not isinstance(adapters, list):
                errors.append(f"{prefix}: 'adapters' must be a list when present")
            else:
                for j, entry in enumerate(adapters):
                    aprefix = f"{prefix}.adapters[{j}]"
                    if not isinstance(entry, dict):
                        errors.append(f"{aprefix}: must be a mapping, got {type(entry).__name__}")
                        continue
                    impl = entry.get("impl")
                    if not impl or not isinstance(impl, str):
                        errors.append(f"{aprefix}: required field 'impl' is missing or not a string")
                    elif ":" not in impl:
                        errors.append(f"{aprefix}: 'impl' must be 'module:ClassName', got {impl!r}")
                    lib = entry.get("lib")
                    if lib is not None and not isinstance(lib, str):
                        errors.append(f"{aprefix}: 'lib' must be a string when present")
    
    return errors

In [ ]:
#| export
def _collect_manifest_warnings(
    data: Any  # Loaded manifest JSON
) -> List[str]:  # Human-readable warning strings (non-failing lints)
    """T23: non-failing manifest lints (warnings, not errors).

    - V4: a single-element `enum` in a config_schema property offers no operator
          choice — the field should be dropped or its domain expanded.
    - V12: quantitative resource fields (`min_gpu_vram_mb` / `recommended_gpu_vram_mb`
          / `min_system_ram_mb`) were dropped by the CR-7 reactive-resource reframe;
          the substrate ignores them, so they are stale dead data.

    Resolves the resources/config_schema location for both v2.0 (nested under
    `code`) and legacy v1.0 (flat) layouts. The `validate` command prints these
    without exiting non-zero (warnings alone don't fail validation).
    """
    warnings: List[str] = []
    if not isinstance(data, dict):
        return warnings
    if data.get("format_version") == "2.0":
        code = data.get("code") if isinstance(data.get("code"), dict) else {}
        res = code.get("resources")
        cs = code.get("config_schema")
        res_path, cs_path = "code.resources", "code.config_schema"
    else:
        res = data.get("resources")
        cs = data.get("config_schema")
        res_path, cs_path = "resources", "config_schema"

    # V12 — dropped quantitative resource fields (CR-7 reframe).
    if isinstance(res, dict):
        for dead_key in ("min_gpu_vram_mb", "recommended_gpu_vram_mb", "min_system_ram_mb"):
            if dead_key in res:
                warnings.append(
                    f"V12: '{res_path}.{dead_key}' is a dropped quantitative resource field "
                    f"(CR-7 reactive-resource reframe); the substrate ignores it — remove it"
                )

    # V4 — single-element enum offers no operator choice.
    if isinstance(cs, dict) and isinstance(cs.get("properties"), dict):
        for field_name, spec in cs["properties"].items():
            if isinstance(spec, dict):
                enum = spec.get("enum")
                if isinstance(enum, list) and len(enum) == 1:
                    warnings.append(
                        f"V4: '{cs_path}.properties.{field_name}.enum' has a single value "
                        f"{enum!r} — drop the field or expand the domain"
                    )
    return warnings

In [ ]:
#| export
def _lint_capability_logging(
    path: Path  # A capability .py file or package directory to scan
) -> tuple:  # (errors, warnings) — lists of human-readable findings
    """T23 (CR-14): lint capability source for `logging.basicConfig` calls.

    The substrate installs the worker\'s root handler
    (`install_worker_diagnostics`) before capability code runs. A capability calling
    `logging.basicConfig(force=True)` DESTROYS that handler (every
    subsequent record silently bypasses the diagnostics store) -> ERROR.
    A plain `basicConfig` call is a no-op once a handler exists — a fragile
    pre-CR-14 idiom that suggests the capability expects to own process logging
    -> WARNING. Directories scan their tree, skipping hidden dirs and
    `tests_manual`/`_proc` (host-side scripts own their own logging).
    """
    import re
    errors: List[str] = []
    warnings: List[str] = []
    files = [path] if path.is_file() else sorted(
        f for f in path.rglob("*.py")
        if not any(part.startswith(".") or part in ("tests_manual", "_proc")
                   for part in f.relative_to(path).parts))
    for f in files:
        try:
            lines = f.read_text(errors="replace").splitlines()
        except OSError as e:
            warnings.append(f"{f}: unreadable ({e})")
            continue
        for i, line in enumerate(lines, 1):
            code = line.split("#", 1)[0]
            if "logging.basicConfig" not in code:
                continue
            # The call may span lines; inspect a small forward window.
            window = " ".join(l.split("#", 1)[0] for l in lines[i - 1:i + 4])
            if re.search(r"force\s*=\s*True", window):
                errors.append(
                    f"{f}:{i}: logging.basicConfig(force=True) destroys the "
                    f"substrate diagnostics handler (CR-14) — use module "
                    f"loggers / self.logger and let the worker own config")
            else:
                warnings.append(
                    f"{f}:{i}: logging.basicConfig is a no-op under the "
                    f"substrate handler (workers are configured by "
                    f"install_worker_diagnostics) — remove it")
    return errors, warnings

In [ ]:
#| export
def _detect_manifest_format(
    path: Path  # File to inspect
) -> Optional[str]:  # 'manifest' | 'capabilities_yaml' | None
    """Auto-detect format: extension for files; directories lint as source."""
    if path.is_dir():
        return "source"
    suffix = path.suffix.lower()
    if suffix == ".json":
        return "manifest"
    if suffix in (".yaml", ".yml"):
        return "capabilities_yaml"
    if suffix == ".py":
        return "source"
    return None

In [ ]:
#| export
@app.command("validate")
def validate_file(
    path:Path=typer.Argument(..., help="Manifest JSON, capabilities.yaml, or capability source (.py / package dir) to validate"),
    format:Optional[str]=typer.Option(
        None, "--format", "-f",
        help="Override format detection: 'manifest', 'capabilities_yaml', or 'source'",
    ),
) -> None:
    """SG-6 + T23: validate a manifest / capabilities.yaml / capability source.
    
    Auto-detects format from the path (`.json` → manifest, `.yaml`/`.yml` →
    capabilities.yaml, `.py` or a directory → source lint). The source lint is
    the CR-14 `logging.basicConfig` gate: `force=True` is an ERROR (it
    destroys the substrate diagnostics handler), a plain call is a WARNING.
    Exits non-zero with a list of validation errors if any check fails.
    """
    if not path.exists():
        typer.echo(f"File not found: {path}", err=True)
        raise typer.Exit(code=1)
    
    fmt = format or _detect_manifest_format(path)
    if fmt is None:
        typer.echo(
            f"Cannot detect format from extension {path.suffix!r}. "
            f"Pass --format manifest, capabilities_yaml, or source.",
            err=True,
        )
        raise typer.Exit(code=1)
    
    warnings: List[str] = []
    try:
        if fmt == "manifest":
            with open(path) as f:
                data = json.load(f)
            errors = _validate_manifest_dict(data)
            warnings = _collect_manifest_warnings(data)
            kind = "manifest"
        elif fmt == "capabilities_yaml":
            with open(path) as f:
                data = yaml.safe_load(f)
            errors = _validate_capabilities_yaml_dict(data)
            kind = "capabilities.yaml"
        elif fmt == "source":
            errors, warnings = _lint_capability_logging(path)
            kind = "capability source"
        else:
            typer.echo(f"Unknown format: {fmt!r}", err=True)
            raise typer.Exit(code=1)
    except (json.JSONDecodeError, yaml.YAMLError) as e:
        typer.echo(f"Parse error in {path}: {e}", err=True)
        raise typer.Exit(code=1)
    
    # T23: warnings (V4 single-element enum, V12 dropped resource fields) are
    # non-failing lints — surface them but don't exit non-zero on warnings alone.
    if warnings:
        typer.echo(f"⚠ {path} ({kind}): {len(warnings)} warning(s)", err=True)
        for w in warnings:
            typer.echo(f"  - {w}", err=True)
    
    if errors:
        typer.echo(f"✗ {path} ({kind}): {len(errors)} error(s)", err=True)
        for err in errors:
            typer.echo(f"  - {err}", err=True)
        raise typer.Exit(code=1)
    
    typer.echo(f"✓ {path} ({kind}): valid{' (with warnings)' if warnings else ''}")

In [ ]:
#| hide
# T23 (CR-14): the basicConfig source lint — force=True is an ERROR, plain
# basicConfig a WARNING, clean files produce nothing, host-side dirs skipped.
import tempfile as _tmp_t23
from pathlib import Path as _P_t23

with _tmp_t23.TemporaryDirectory() as _td:
    root = _P_t23(_td)
    (root / "pkg").mkdir()
    (root / "pkg" / "good.py").write_text(
        "import logging\nlogger = logging.getLogger(__name__)\n")
    (root / "pkg" / "warn.py").write_text(
        "import logging\nlogging.basicConfig(level=logging.INFO)\n")
    (root / "pkg" / "bad.py").write_text(
        "import logging\nlogging.basicConfig(\n    level=logging.INFO,\n    force=True,\n)\n")
    (root / "tests_manual").mkdir()
    (root / "tests_manual" / "script.py").write_text(
        "import logging\nlogging.basicConfig(force=True)\n")  # host-side: skipped

    errors, warnings = _lint_capability_logging(root)
    assert len(errors) == 1 and "bad.py:2" in errors[0] and "force=True" in errors[0], errors
    assert len(warnings) == 1 and "warn.py:2" in warnings[0], warnings

    # Single-file mode + comment-only mentions don't fire.
    (root / "pkg" / "commented.py").write_text(
        "# logging.basicConfig is discouraged\nx = 1\n")
    e2, w2 = _lint_capability_logging(root / "pkg" / "commented.py")
    assert e2 == [] and w2 == []

print("T23 basicConfig source lint: PASS")

In [ ]:
#| hide
# SG-6 regression: structural validators flag obvious malformed inputs and
# pass through well-formed ones. Tests the validators directly rather than
# the typer command (which is exercised via cjm-ctl validate at the CLI).
#
# v2.0 nested dispatch (format_version == "2.0"). The legacy v1.0 flat
# dispatch + its validator were removed at SG-48.

# ─── v2.0 nested layout (CR-8) ───
good_v2 = {
    "format_version": "2.0",
    "install": {
        "python_path": "/tmp/envs/whisper/bin/python",
        "conda_env": "whisper",
        "db_path": "/data/whisper.db",
        "env_vars": {"HF_HOME": "/models/hf"},
        "installed_at": "2026-05-22T12:00:00+00:00",
        "installer_version": "cjm-ctl 0.0.30",
        "package_source": "git+https://github.com/cj-mills/cjm-capability-whisper.git",
    },
    "code": {
        "name": "whisper-local",
        "version": "1.0.0",
        "description": "Local Whisper-based speech-to-text capability.",
        "module": "cjm_capability_whisper.capability",
        "class": "WhisperCapability",
        "resources": {
            "requires_gpu": True,
            "platforms": ["linux-x64"],
            "accelerators": ["cuda"],
        },
        "config_schema": {"type": "object", "properties": {"model": {"type": "string"}}},
        "regenerated_at": "2026-05-22T12:00:01+00:00",
    },
    "drift_tracking": {"config_schema_hash": "sha256:abc"},
    "overrides": {},
}
errors = _validate_manifest_dict(good_v2)
assert errors == [], f"valid v2.0 manifest reported errors: {errors}"

# Missing required section
errors = _validate_manifest_dict({"format_version": "2.0", "install": good_v2["install"]})
assert any("'code'" in e and "missing" in e for e in errors), \
    f"missing 'code' section must be flagged; got {errors}"

# Missing required code.* field
import copy as _copy
bad_v2 = _copy.deepcopy(good_v2)
del bad_v2["code"]["module"]
errors = _validate_manifest_dict(bad_v2)
assert any("'code.module'" in e and "missing" in e for e in errors), \
    f"missing code.module must be flagged; got {errors}"

# Missing required install.python_path
bad_v2 = _copy.deepcopy(good_v2)
del bad_v2["install"]["python_path"]
errors = _validate_manifest_dict(bad_v2)
assert any("'install.python_path'" in e and "missing" in e for e in errors), \
    f"missing install.python_path must be flagged; got {errors}"

# Bad install.env_vars type
bad_v2 = _copy.deepcopy(good_v2)
bad_v2["install"]["env_vars"] = "not an object"
errors = _validate_manifest_dict(bad_v2)
assert any("'install.env_vars'" in e for e in errors)

# Bad drift_tracking.config_schema_hash type
bad_v2 = _copy.deepcopy(good_v2)
bad_v2["drift_tracking"]["config_schema_hash"] = 12345
errors = _validate_manifest_dict(bad_v2)
assert any("'drift_tracking.config_schema_hash'" in e for e in errors)

# Resources type-checking on nested layout
bad_v2 = _copy.deepcopy(good_v2)
bad_v2["code"]["resources"]["requires_gpu"] = "yes"
errors = _validate_manifest_dict(bad_v2)
assert any("requires_gpu" in e and "boolean" in e for e in errors)

# Unrecognized format_version rejects loud
errors = _validate_manifest_dict({"format_version": "3.0", "install": {}, "code": {}})
assert any("unrecognized format_version" in e for e in errors), \
    f"unknown format_version must be flagged; got {errors}"

# Missing format_version now also rejects loud (the v1.0 flat shim was removed at SG-48).
errors = _validate_manifest_dict({"name": "x", "version": "1.0.0", "module": "x.capability"})
assert any("unrecognized format_version" in e for e in errors), \
    f"missing format_version must be flagged; got {errors}"

# Non-dict at root.
errors = _validate_manifest_dict(["this", "is", "wrong"])
assert any("must be a JSON object" in e for e in errors)

# Bad code.config_schema shape.
_bad_cs = _copy.deepcopy(good_v2)
_bad_cs["code"]["config_schema"] = "not an object"
assert any("config_schema" in e for e in _validate_manifest_dict(_bad_cs))

# ─── capabilities.yaml validator (unchanged) ───
good_yaml = {
    "capabilities": [
        {"name": "a", "env_name": "env-a", "package": "pkg-a", "python_version": "3.12"},
        {"name": "b", "env_name": "env-b", "package": "pkg-b", "env_file": "b.yml",
         "interface_libs": ["lib1", "lib2"]},
    ]
}
errors = _validate_capabilities_yaml_dict(good_yaml)
assert errors == [], f"valid capabilities.yaml reported errors: {errors}"

# Missing top-level 'capabilities'
errors = _validate_capabilities_yaml_dict({})
assert any("'capabilities' is missing" in e for e in errors)

# Duplicate names
errors = _validate_capabilities_yaml_dict({
    "capabilities": [
        {"name": "dup", "env_name": "e1", "package": "p1", "python_version": "3.12"},
        {"name": "dup", "env_name": "e2", "package": "p2", "python_version": "3.12"},
    ]
})
assert any("duplicate capability name" in e for e in errors)

# Neither env_file nor python_version
errors = _validate_capabilities_yaml_dict({
    "capabilities": [{"name": "x", "env_name": "e", "package": "p"}]
})
assert any("'env_file' or 'python_version'" in e for e in errors)

# Adapters key (stage 6 J10): well-formed passes; malformed flagged loudly
errors = _validate_capabilities_yaml_dict({
    "capabilities": [{"name": "g", "env_name": "e", "package": "p", "python_version": "3.12",
                 "adapters": [{"lib": "some-adapter-lib", "impl": "mod.sub:Cls"}]}]
})
assert errors == [], f"valid adapters entry reported errors: {errors}"
errors = _validate_capabilities_yaml_dict({
    "capabilities": [{"name": "g", "env_name": "e", "package": "p", "python_version": "3.12",
                 "adapters": [{"lib": "some-lib", "impl": "no-colon"}, "not-a-dict"]}]
})
assert any("'impl' must be 'module:ClassName'" in e for e in errors)
assert any("must be a mapping" in e for e in errors)
errors = _validate_capabilities_yaml_dict({
    "capabilities": [{"name": "g", "env_name": "e", "package": "p", "python_version": "3.12",
                 "adapters": {"impl": "m:C"}}]
})
assert any("'adapters' must be a list" in e for e in errors)

# Format detection from extension
assert _detect_manifest_format(Path("a.json")) == "manifest"
assert _detect_manifest_format(Path("capabilities.yaml")) == "capabilities_yaml"
assert _detect_manifest_format(Path("capabilities.yml")) == "capabilities_yaml"
assert _detect_manifest_format(Path("README.md")) is None

print("SG-6 validate (v2.0 + capabilities.yaml): PASS")

# ─── T23 first-wave validate gates ───────────────────────────────────────
# V1 (whitespace-only required field), V4 (single-element enum WARNING),
# V12 (dropped quantitative resource field WARNING), worker-env default
# template check (ERROR on unknown placeholder), and the T31 cross-check
# that the renamed ${CJM_CAPABILITY_DATA_DIR} is valid while old ${CJM_DATA_DIR}
# is now an unknown placeholder.

_t23_v2 = {
    "format_version": "2.0",
    "install": {"python_path": "/tmp/envs/x/bin/python"},
    "code": {
        "name": "cjm-x-capability",
        "version": "0.0.1",
        "description": "X capability.",
        "module": "cjm_x_capability.capability",
        "class": "XCapability",
    },
}
assert _validate_manifest_dict(_t23_v2) == [], _validate_manifest_dict(_t23_v2)

# V1 — whitespace-only description now rejected (was: only "" rejected).
_ws_v2 = {**_t23_v2, "code": {**_t23_v2["code"], "description": "   "}}
assert any("code.description" in e and ("missing" in e or "empty" in e)
           for e in _validate_manifest_dict(_ws_v2)), _validate_manifest_dict(_ws_v2)

# Worker-env template check — unknown placeholder is a capability-author bug (ERROR).
_bad_we = {**_t23_v2, "code": {**_t23_v2["code"],
    "worker_env": [{"name": "HF_HOME", "default": "${NOPE}/hf"}]}}
assert any("worker_env" in e and "NOPE" in e for e in _validate_manifest_dict(_bad_we)), \
    _validate_manifest_dict(_bad_we)
# Allowed placeholders (incl. the T31-renamed CJM_CAPABILITY_DATA_DIR) validate clean.
_ok_we = {**_t23_v2, "code": {**_t23_v2["code"], "worker_env": [
    {"name": "HF_HOME", "default": "${CJM_MODELS_DIR}/huggingface"},
    {"name": "NLTK_DATA", "default": "${CAPABILITY_DATA_DIR}/nltk_data"},
    {"name": "STORE", "default": "${CJM_CAPABILITY_DATA_DIR}/store"},
    {"name": "CUDA_VISIBLE_DEVICES", "default": "0"},
]}}
assert _validate_manifest_dict(_ok_we) == [], _validate_manifest_dict(_ok_we)
# T31: the OLD ${CJM_DATA_DIR} placeholder is no longer in the vocabulary.
_old_we = {**_t23_v2, "code": {**_t23_v2["code"],
    "worker_env": [{"name": "FOO", "default": "${CJM_DATA_DIR}/foo"}]}}
assert any("worker_env" in e and "CJM_DATA_DIR" in e for e in _validate_manifest_dict(_old_we)), \
    _validate_manifest_dict(_old_we)
# worker_env must be a list
assert any("worker_env" in e and "list" in e for e in
           _validate_manifest_dict({**_t23_v2, "code": {**_t23_v2["code"], "worker_env": {}}}))

# V4 — single-element enum WARNS (non-failing); multi-value enum does not.
_v4 = {**_t23_v2, "code": {**_t23_v2["code"], "config_schema": {"type": "object", "properties": {
    "device": {"type": "string", "enum": ["cuda"]},
    "model": {"type": "string", "enum": ["a", "b"]}}}}}
_w = _collect_manifest_warnings(_v4)
assert any("V4" in w and "device" in w for w in _w), _w
assert not any("model" in w for w in _w), f"2-value enum must not warn: {_w}"
assert _validate_manifest_dict(_v4) == [], _validate_manifest_dict(_v4)  # warning != error

# V12 — dropped quantitative resource field WARNS.
_v12 = {**_t23_v2, "code": {**_t23_v2["code"],
    "resources": {"requires_gpu": True, "min_gpu_vram_mb": 4096}}}
assert any("V12" in w and "min_gpu_vram_mb" in w for w in _collect_manifest_warnings(_v12))

# Clean manifests produce no warnings.
assert _collect_manifest_warnings(_t23_v2) == []

print("T23 first-wave validate gates (V1/V4/V12 + worker-env template + T31 vocab): PASS")

In [ ]:
#| export
# ----------------------------------------------------------------
# Secret management (CR-12)
# ----------------------------------------------------------------

def _open_secret_store():
    """Open the project-local LocalSecretStore at <data_dir>/secrets (CR-12)."""
    from cjm_substrate.core.secret_store import LocalSecretStore
    cfg = get_config()
    data_dir = getattr(cfg, "data_dir", None)
    secrets_dir = (data_dir / "secrets") if data_dir is not None else None
    return LocalSecretStore(secrets_dir)


@app.command("set-secret")
def set_secret(
    capability_name: str = typer.Argument(..., help="Capability name (manifest 'name', e.g. my-api-capability)"),
    key: str = typer.Argument(..., help="Secret key = the env-var name the worker reads (e.g. MY_API_KEY)"),
    value: Optional[str] = typer.Option(None, "--value", help="Secret value (omit to be prompted with hidden input)"),
    scope: Optional[str] = typer.Option(None, "--scope", help="Reserved multi-user scope (default: single-user)"),
):
    """Store a capability secret in the project-local SecretStore (CR-12).

    The value is written to <data_dir>/secrets/secrets.json (0600) — never to
    capabilities.yaml, manifests, or the config store. Capabilities read it from their
    worker env at spawn. Omit --value to be prompted (hidden input) so the
    secret stays out of shell history. After setting, reload the capability (or
    restart the host) so its worker respawns with the new env — the GUI /
    CapabilityManager.set_capability_secret do this automatically.
    """
    store = _open_secret_store()
    if value is None:
        value = typer.prompt(f"Value for {capability_name}/{key}", hide_input=True)
    store.set_secret(capability_name, key, value, scope=scope)
    typer.echo(f"Stored secret {key!r} for capability {capability_name!r} at {store.path} (0600).")
    typer.echo("Reload the capability (or restart the host) so its worker respawns with the new env.")


@app.command("list-secrets")
def list_secrets(
    capability_name: str = typer.Argument(..., help="Capability name to list secret KEY NAMES for"),
    scope: Optional[str] = typer.Option(None, "--scope", help="Reserved multi-user scope"),
):
    """List the secret KEY NAMES stored for a capability — never the values (CR-12)."""
    store = _open_secret_store()
    keys = store.list_keys(capability_name, scope=scope)
    if not keys:
        typer.echo(f"No secrets stored for {capability_name!r}.")
        return
    typer.echo(f"Secrets for {capability_name!r} ({len(keys)}):")
    for k in keys:
        typer.echo(f"  - {k}")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()